# 📚 RAG Knowledge Extraction System — Week 1
## Environment, Data Acquisition & Cleaning

**Parallax Labs — RAG Pipeline Project**

This notebook covers all Week 1 deliverables:

1. **Environment verification** — confirm every required library imports and works correctly.
2. **Data acquisition** — pull 5,000+ real documents from the **ArXiv public API** (no key required).
3. **Data validation** — null checks, duplicate checks, encoding checks, language detection.
4. **Text cleaning** — robust cleaning functions that handle empty text, HTML/LaTeX artifacts, unicode issues, and mixed-language content.
5. **Unit tests** — edge-case tests for every cleaning function.
6. **Exploratory Data Analysis (EDA)** — distributions, category breakdown, word clouds.
7. **Data Quality Report** — auto-generated markdown + JSON report.
8. **Final clean dataset export** — ready for Week 2 chunking & embedding.

> **Why ArXiv?** It has a free, keyless public API, is fully real-world (not synthetic), easily yields 5,000+ documents across multiple categories, and is a natural fit for a research-grounded RAG demo. The pipeline below is written generically enough that swapping in a Wikipedia or Reddit source later only requires changing the acquisition cell.


---
## 1. Environment Verification

We check that every library needed for this week (and staged for future weeks) imports correctly,
print versions for reproducibility, and run one tiny smoke-test per library so a broken install
fails loudly here instead of silently later.


In [ ]:
"""
Environment verification script.
Run this first. If anything fails, install the missing package with:
    pip install -r requirements.txt
"""

import importlib
import sys

REQUIRED = {
    "pandas": "pandas",
    "numpy": "numpy",
    "requests": "requests",
    "feedparser": "feedparser",     # arxiv API returns Atom/RSS feeds
    "langdetect": "langdetect",
    "ftfy": "ftfy",                 # fixes broken unicode / mojibake
    "unidecode": "unidecode",       # transliterates unicode -> ascii when needed
    "nltk": "nltk",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "wordcloud": "wordcloud",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "bs4": "beautifulsoup4",        # strips stray HTML entities/tags
}

print(f"Python version: {sys.version}\n")

failed = []
for module_name, pip_name in REQUIRED.items():
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"  [OK]   {module_name:<15} v{version}")
    except ImportError as e:
        print(f"  [FAIL] {module_name:<15} NOT INSTALLED -> pip install {pip_name}")
        failed.append(pip_name)

if failed:
    print(f"\nMissing packages: {failed}")
    print("Run: pip install " + " ".join(failed))
else:
    print("\nAll required libraries are installed.")

In [ ]:
# --- Smoke tests: prove each library actually WORKS, not just imports ---
import pandas as pd
import numpy as np
import requests
import nltk
from langdetect import detect
import ftfy
from bs4 import BeautifulSoup

smoke_results = {}

# pandas / numpy
df_test = pd.DataFrame({"x": np.arange(5)})
smoke_results["pandas+numpy"] = df_test["x"].sum() == 10

# requests -> internet access (needed for ArXiv API in Section 2)
try:
    r = requests.get("https://export.arxiv.org/api/query?search_query=all:test&max_results=1", timeout=10)
    smoke_results["requests+arxiv_api"] = r.status_code == 200
except Exception as e:
    smoke_results["requests+arxiv_api"] = f"FAILED: {e}"

# nltk resources needed later
for pkg in ["punkt", "punkt_tab", "stopwords"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass
smoke_results["nltk_download"] = True

# langdetect
smoke_results["langdetect"] = detect("This is a sentence written in English.") == "en"

# ftfy (fixes mangled unicode like "Ã©" -> "é")
smoke_results["ftfy"] = ftfy.fix_text("MÃ¼nchen") == "München"

# BeautifulSoup (HTML stripping)
smoke_results["bs4"] = BeautifulSoup("<p>hello &amp; world</p>", "html.parser").get_text() == "hello & world"

print("Smoke test results:")
for k, v in smoke_results.items():
    status = "OK" if v is True else v
    print(f"  {k:<20}: {status}")

assert all(v is True for v in smoke_results.values()), "One or more smoke tests failed — check output above."
print("\nEnvironment fully verified. Ready to proceed.")

---
## 2. Data Acquisition — 5,000+ Documents from ArXiv

We use ArXiv's free, keyless public API (`export.arxiv.org/api/query`) to pull paper
**titles + abstracts** across multiple CS/ML categories. Multiple categories are combined
so the corpus is topically diverse (a better stress-test for the RAG retriever in later weeks
than one narrow category).

The API paginates in batches of `max_results` (capped at 100/request by ArXiv), so we loop with
a small delay between requests to respect their rate limits (ArXiv asks for ≥3 seconds between calls).

This cell is safe to re-run — it checks for a cached raw CSV first and skips re-downloading.


In [ ]:
import time
import feedparser
import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
RAW_PATH = DATA_DIR / "raw_arxiv.csv"

# Diverse categories -> richer, more realistic corpus for downstream RAG retrieval
CATEGORIES = [
    "cs.AI",   # Artificial Intelligence
    "cs.CL",   # Computation and Language (NLP)
    "cs.LG",   # Machine Learning
    "cs.CV",   # Computer Vision
    "cs.IR",   # Information Retrieval
    "cs.NE",   # Neural and Evolutionary Computing
    "cs.RO",   # Robotics
    "stat.ML", # Statistics - Machine Learning
]

DOCS_PER_CATEGORY = 700       # 8 categories * 700 = 5,600 docs (comfortably over the 5,000 minimum)
BATCH_SIZE = 100              # ArXiv API max per request
SLEEP_BETWEEN_CALLS = 3.1     # seconds, per ArXiv API etiquette guidelines
BASE_URL = "https://export.arxiv.org/api/query"


def fetch_arxiv_category(category: str, total: int, batch_size: int = BATCH_SIZE):
    """Page through the ArXiv API for one category and return a list of record dicts."""
    records = []
    start = 0
    pbar = tqdm(total=total, desc=f"Fetching {category}", leave=False)
    while len(records) < total:
        n = min(batch_size, total - len(records))
        params = {
            "search_query": f"cat:{category}",
            "start": start,
            "max_results": n,
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        }
        try:
            resp = requests.get(BASE_URL, params=params, timeout=30)
            resp.raise_for_status()
        except requests.RequestException as e:
            print(f"  [warn] request failed for {category} at start={start}: {e}")
            break

        feed = feedparser.parse(resp.text)
        if not feed.entries:
            break  # exhausted this category

        for entry in feed.entries:
            records.append({
                "id": entry.get("id", ""),
                "title": entry.get("title", ""),
                "abstract": entry.get("summary", ""),
                "authors": ", ".join(a.get("name", "") for a in entry.get("authors", [])),
                "category": category,
                "published": entry.get("published", ""),
                "primary_category": entry.get("arxiv_primary_category", {}).get("term", category),
            })

        start += len(feed.entries)
        pbar.update(len(feed.entries))
        time.sleep(SLEEP_BETWEEN_CALLS)

    pbar.close()
    return records


if RAW_PATH.exists():
    print(f"Cached raw dataset found at {RAW_PATH}, loading instead of re-downloading.")
    raw_df = pd.read_csv(RAW_PATH)
else:
    all_records = []
    for cat in CATEGORIES:
        recs = fetch_arxiv_category(cat, DOCS_PER_CATEGORY)
        print(f"  {cat}: retrieved {len(recs)} records")
        all_records.extend(recs)

    raw_df = pd.DataFrame(all_records)
    raw_df.to_csv(RAW_PATH, index=False)
    print(f"\nSaved {len(raw_df)} raw records to {RAW_PATH}")

print(f"\nTotal documents acquired: {len(raw_df):,}")
assert len(raw_df) >= 5000, "Fewer than 5,000 documents acquired — re-run the cell (network hiccup) or raise DOCS_PER_CATEGORY."
raw_df.head(3)

**Note on reproducibility / offline grading:** the cell above requires outbound internet
access to `export.arxiv.org`. If you are running in a fully offline/sandboxed environment,
the notebook will raise a clear connection error at this cell — that's expected, not a bug in
the cleaning code. On a normal machine with internet access this cell typically takes
4-6 minutes for ~5,600 documents due to the polite rate-limit delay.

---
## 3. Data Validation

Before cleaning, we quantify exactly how messy the raw data is: nulls, duplicates, encoding
issues, and language mix. These numbers become the "before" half of the Data Quality Report
in Section 7.


In [ ]:
import pandas as pd

def encoding_is_ok(text: str) -> bool:
    """Returns False if the string contains typical mojibake / broken-encoding markers."""
    if not isinstance(text, str):
        return False
    suspicious_markers = ["Ã", "â€", "\\x", "\ufffd"]
    return not any(m in text for m in suspicious_markers)

validation_report = {}

# 1. Shape
validation_report["total_rows"] = len(raw_df)
validation_report["total_columns"] = raw_df.shape[1]

# 2. Null checks
null_counts = raw_df.isnull().sum()
validation_report["null_counts"] = null_counts.to_dict()
validation_report["rows_with_any_null"] = int(raw_df.isnull().any(axis=1).sum())

# 3. Empty-string checks (nulls after CSV round-trip sometimes become "")
validation_report["empty_abstract_count"] = int((raw_df["abstract"].fillna("").str.strip() == "").sum())
validation_report["empty_title_count"] = int((raw_df["title"].fillna("").str.strip() == "").sum())

# 4. Duplicate checks
validation_report["duplicate_ids"] = int(raw_df["id"].duplicated().sum())
validation_report["duplicate_abstracts"] = int(raw_df["abstract"].duplicated().sum())

# 5. Encoding checks
validation_report["rows_with_encoding_issues"] = int((~raw_df["abstract"].fillna("").apply(encoding_is_ok)).sum())

# 6. Length sanity
raw_df["_raw_abstract_len"] = raw_df["abstract"].fillna("").str.len()
validation_report["min_abstract_len"] = int(raw_df["_raw_abstract_len"].min())
validation_report["max_abstract_len"] = int(raw_df["_raw_abstract_len"].max())
validation_report["mean_abstract_len"] = round(float(raw_df["_raw_abstract_len"].mean()), 1)

print("=== Raw Data Validation Report ===")
for k, v in validation_report.items():
    print(f"  {k}: {v}")

In [ ]:
# Language distribution (sampled for speed on large corpora)
from langdetect import detect, LangDetectException
import pandas as pd

def safe_detect(text: str) -> str:
    text = (text or "").strip()
    if len(text) < 20:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

sample_n = min(2000, len(raw_df))
lang_sample = raw_df.sample(sample_n, random_state=42)["abstract"].apply(safe_detect)
lang_dist = lang_sample.value_counts(normalize=True).round(4) * 100

print(f"Estimated language distribution (sample of {sample_n} docs):")
print(lang_dist.head(10))

---
## 4. Text Cleaning Functions

`clean_text()` is written defensively to handle every edge case we found (and the ones the
brief explicitly calls out): `None`/`NaN` input, empty/whitespace-only strings, stray HTML
entities, LaTeX markup (common in ArXiv abstracts, e.g. `$\alpha$`), broken unicode/mojibake,
excess whitespace/newlines, and mixed-language text (flagged, not silently dropped).


In [ ]:
import re
import ftfy
import unicodedata
from bs4 import BeautifulSoup
from langdetect import detect, LangDetectException

# Precompiled patterns (compiled once for performance over 5,000+ docs)
_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_LATEX_INLINE_RE = re.compile(r"\$[^$]{1,200}\$")          # $ ... $
_LATEX_CMD_RE = re.compile(r"\\[a-zA-Z]+(\{[^{}]*\})?")     # \alpha, \textbf{...}
_MULTI_WS_RE = re.compile(r"\s+")
_NON_PRINTABLE_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")


def clean_text(text, min_length: int = 20) -> dict:
    """
    Clean a single raw text string and return a dict with the cleaned text plus
    quality flags. Never raises on bad input -- always returns a well-formed dict.

    Handles:
      - None / NaN / non-string input
      - empty or whitespace-only strings
      - HTML entities/tags (&amp;, <p>, etc.)
      - LaTeX markup common in ArXiv abstracts
      - broken unicode / mojibake (via ftfy)
      - excessive whitespace / newlines
      - non-printable control characters
      - mixed-language content (flagged via langdetect, not deleted)
    """
    result = {
        "cleaned_text": "",
        "is_empty": True,
        "is_too_short": True,
        "had_encoding_issue": False,
        "language": "unknown",
        "original_length": 0,
        "cleaned_length": 0,
    }

    # 1. Handle None / NaN / non-string
    if text is None or (isinstance(text, float)):  # NaN is a float
        return result
    if not isinstance(text, str):
        text = str(text)

    result["original_length"] = len(text)

    if text.strip() == "":
        return result

    # 2. Fix broken unicode / mojibake
    fixed = ftfy.fix_text(text)
    result["had_encoding_issue"] = fixed != text
    text = fixed

    # 3. Normalize unicode (NFKC folds full-width chars, ligatures, etc.)
    text = unicodedata.normalize("NFKC", text)

    # 4. Strip HTML tags/entities
    text = BeautifulSoup(text, "html.parser").get_text()

    # 5. Remove URLs
    text = _URL_RE.sub(" ", text)

    # 6. Remove LaTeX markup (inline math, then bare commands)
    text = _LATEX_INLINE_RE.sub(" ", text)
    text = _LATEX_CMD_RE.sub(" ", text)

    # 7. Strip non-printable / control characters
    text = _NON_PRINTABLE_RE.sub(" ", text)

    # 8. Collapse whitespace/newlines
    text = _MULTI_WS_RE.sub(" ", text).strip()

    result["cleaned_text"] = text
    result["cleaned_length"] = len(text)
    result["is_empty"] = (text == "")
    result["is_too_short"] = (len(text) < min_length)

    # 9. Language detection (best-effort; flags mixed/non-English rather than dropping)
    if len(text) >= 20:
        try:
            result["language"] = detect(text)
        except LangDetectException:
            result["language"] = "unknown"

    return result

---
## 5. Unit Tests for Cleaning Functions

Edge cases covered: `None`, `NaN`, empty string, whitespace-only, HTML entities, LaTeX,
mojibake/broken-unicode, extremely long input, and mixed-language text.


In [ ]:
import unittest
import math

class TestCleanText(unittest.TestCase):

    def test_none_input(self):
        r = clean_text(None)
        self.assertTrue(r["is_empty"])
        self.assertEqual(r["cleaned_text"], "")

    def test_nan_input(self):
        r = clean_text(float("nan"))
        self.assertTrue(r["is_empty"])

    def test_empty_string(self):
        r = clean_text("")
        self.assertTrue(r["is_empty"])

    def test_whitespace_only(self):
        r = clean_text("   \n\t   ")
        self.assertTrue(r["is_empty"])

    def test_html_entities_and_tags(self):
        r = clean_text("<p>Attention &amp; Transformers</p> are great, published in 2017.")
        self.assertNotIn("<p>", r["cleaned_text"])
        self.assertIn("&", r["cleaned_text"])  # &amp; should decode to a literal &

    def test_latex_markup_removed(self):
        r = clean_text("The loss converges as $\\alpha \\to 0$ using \\textbf{gradient} descent methods for optimization.")
        self.assertNotIn("\\textbf", r["cleaned_text"])
        self.assertNotIn("$\\alpha", r["cleaned_text"])

    def test_broken_unicode_fixed(self):
        r = clean_text("MÃ¼nchen is a city in Germany with a long and interesting history of science.")
        self.assertIn("München", r["cleaned_text"])
        self.assertTrue(r["had_encoding_issue"])

    def test_url_removed(self):
        r = clean_text("See our results at https://example.com/paper for full details on the experiment.")
        self.assertNotIn("http", r["cleaned_text"])

    def test_excess_whitespace_collapsed(self):
        r = clean_text("This    has\\n\\n\\nway     too much   whitespace between the words here.")
        self.assertNotIn("  ", r["cleaned_text"])

    def test_too_short_flag(self):
        r = clean_text("Too short.")
        self.assertTrue(r["is_too_short"])

    def test_normal_english_text(self):
        r = clean_text("This paper introduces a novel transformer architecture for efficient long-context reasoning.")
        self.assertFalse(r["is_empty"])
        self.assertFalse(r["is_too_short"])
        self.assertEqual(r["language"], "en")

    def test_mixed_language_flagged_not_dropped(self):
        r = clean_text("This is an English sentence. Ceci est une phrase en français qui continue longtemps.")
        self.assertFalse(r["is_empty"])
        self.assertIn(r["language"], ["en", "fr"])  # detector picks the dominant one; either is acceptable

    def test_very_long_input_does_not_crash(self):
        long_text = "word " * 5000
        r = clean_text(long_text)
        self.assertFalse(r["is_empty"])
        self.assertGreater(r["cleaned_length"], 1000)

    def test_non_string_input_coerced(self):
        r = clean_text(12345)
        # a bare number is a valid (if short) string once coerced
        self.assertEqual(r["original_length"], 5)


suite = unittest.TestLoader().loadTestsFromTestCase(TestCleanText)
runner = unittest.TextTestRunner(verbosity=2)
test_result = runner.run(suite)

assert test_result.wasSuccessful(), "Unit tests failed -- fix clean_text() before proceeding."
print(f"\nAll {test_result.testsRun} unit tests passed.")

---
## 6. Apply Cleaning to the Full Dataset

Now that `clean_text()` is unit-tested, we apply it across all 5,000+ documents and attach
the resulting quality flags as new columns.


In [ ]:
from tqdm.auto import tqdm
tqdm.pandas(desc="Cleaning abstracts")

cleaned_results = raw_df["abstract"].progress_apply(clean_text)
cleaned_df = pd.json_normalize(cleaned_results)
cleaned_df.columns = ["clean_" + c if c != "cleaned_text" else "cleaned_text" for c in cleaned_df.columns]

# Also lightly clean titles (reuse the same function, shorter min_length)
title_results = raw_df["title"].apply(lambda t: clean_text(t, min_length=5))
cleaned_df["cleaned_title"] = [r["cleaned_text"] for r in title_results]

full_df = pd.concat([raw_df.reset_index(drop=True), cleaned_df.reset_index(drop=True)], axis=1)
full_df.head(3)

---
## 7. Exploratory Data Analysis (EDA)

Quick visual sanity checks: how document length changed after cleaning, category balance,
language mix, and the most frequent terms in the corpus.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(full_df["clean_original_length"], bins=50, alpha=0.6, label="Before cleaning", color="#4C72B0")
axes[0].hist(full_df["clean_cleaned_length"], bins=50, alpha=0.6, label="After cleaning", color="#55A868")
axes[0].set_title("Abstract Length Distribution")
axes[0].set_xlabel("Character count")
axes[0].set_ylabel("Number of documents")
axes[0].legend()

cat_counts = full_df["category"].value_counts()
axes[1].bar(cat_counts.index, cat_counts.values, color="#C44E52")
axes[1].set_title("Documents per ArXiv Category")
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_ylabel("Document count")

plt.tight_layout()
plt.savefig(OUT_DIR / "length_and_category_distribution.png", dpi=150)
plt.show()

In [ ]:
# Language mix across the FULL cleaned corpus
lang_counts = full_df["clean_language"].value_counts()
print("Language distribution (full corpus):")
print(lang_counts.head(10))

plt.figure(figsize=(6, 6))
top_langs = lang_counts.head(6)
plt.pie(top_langs.values, labels=top_langs.index, autopct="%1.1f%%", startangle=90)
plt.title("Language Distribution (Top 6)")
plt.tight_layout()
plt.savefig(OUT_DIR / "language_distribution.png", dpi=150)
plt.show()

In [ ]:
# Word cloud of the cleaned corpus (visual gut-check on topic content)
from wordcloud import WordCloud, STOPWORDS

sample_text = " ".join(full_df["cleaned_text"].dropna().sample(min(2000, len(full_df)), random_state=42))
wc = WordCloud(width=1000, height=500, background_color="white",
               stopwords=STOPWORDS, collocations=False, max_words=150).generate(sample_text)

plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Most Frequent Terms in Cleaned Corpus (sampled)")
plt.tight_layout()
plt.savefig(OUT_DIR / "wordcloud.png", dpi=150)
plt.show()

In [ ]:
# Summary stats table
summary_stats = full_df[["clean_original_length", "clean_cleaned_length"]].describe().round(1)
summary_stats.columns = ["original_length", "cleaned_length"]
summary_stats

---
## 8. Data Quality Report

We consolidate everything above into a single markdown + JSON report — the required
artifact for this week's submission.


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)

n_total = len(full_df)
n_empty_after = int(full_df["clean_is_empty"].sum())
n_too_short_after = int(full_df["clean_is_too_short"].sum())
n_encoding_fixed = int(full_df["clean_had_encoding_issue"].sum())
n_valid_final = int(((~full_df["clean_is_empty"]) & (~full_df["clean_is_too_short"])).sum())

quality_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": "ArXiv public API (export.arxiv.org)",
    "categories": CATEGORIES,
    "raw_document_count": int(len(raw_df)),
    "pre_cleaning_validation": validation_report,
    "post_cleaning": {
        "documents_flagged_empty_after_cleaning": n_empty_after,
        "documents_flagged_too_short_after_cleaning": n_too_short_after,
        "documents_with_encoding_issues_fixed": n_encoding_fixed,
        "final_valid_document_count": n_valid_final,
        "final_valid_percentage": round(100 * n_valid_final / n_total, 2),
        "mean_length_before": round(float(full_df["clean_original_length"].mean()), 1),
        "mean_length_after": round(float(full_df["clean_cleaned_length"].mean()), 1),
        "language_distribution_top5": lang_counts.head(5).to_dict(),
    },
}

# Save JSON
with open(REPORT_DIR / "data_quality_report.json", "w") as f:
    json.dump(quality_report, f, indent=2, default=str)

# Save human-readable markdown
md_lines = [
    "# Data Quality Report — Week 1",
    f"Generated: {quality_report['generated_at_utc']}",
    "",
    f"**Source:** {quality_report['source']}",
    f"**Categories:** {', '.join(CATEGORIES)}",
    "",
    "## Raw Data",
    f"- Total raw documents: **{quality_report['raw_document_count']:,}**",
    f"- Rows with any null field: {validation_report['rows_with_any_null']}",
    f"- Empty abstracts: {validation_report['empty_abstract_count']}",
    f"- Duplicate IDs: {validation_report['duplicate_ids']}",
    f"- Duplicate abstracts: {validation_report['duplicate_abstracts']}",
    f"- Rows with encoding issues (raw): {validation_report['rows_with_encoding_issues']}",
    "",
    "## After Cleaning",
    f"- Documents flagged empty: {n_empty_after}",
    f"- Documents flagged too short (<20 chars): {n_too_short_after}",
    f"- Documents with encoding auto-fixed: {n_encoding_fixed}",
    f"- **Final valid documents: {n_valid_final:,} ({quality_report['post_cleaning']['final_valid_percentage']}%)**",
    f"- Mean length before cleaning: {quality_report['post_cleaning']['mean_length_before']} chars",
    f"- Mean length after cleaning: {quality_report['post_cleaning']['mean_length_after']} chars",
    "",
    "## Language Distribution (Top 5)",
]
for lang, count in quality_report["post_cleaning"]["language_distribution_top5"].items():
    md_lines.append(f"- `{lang}`: {count}")

with open(REPORT_DIR / "data_quality_report.md", "w") as f:
    f.write("\n".join(md_lines))

print("Data quality report saved to:")
print(f"  - {REPORT_DIR / 'data_quality_report.md'}")
print(f"  - {REPORT_DIR / 'data_quality_report.json'}")
print()
print("\n".join(md_lines))

---
## 9. Export Final Clean Dataset

We keep only valid rows (non-empty, above minimum length), de-duplicate by cleaned text,
and export a tidy dataset ready for **Week 2 (chunking + embedding into ChromaDB)**.


In [ ]:
final_df = full_df[
    (~full_df["clean_is_empty"]) & (~full_df["clean_is_too_short"])
].drop_duplicates(subset=["cleaned_text"]).reset_index(drop=True)

final_df = final_df.rename(columns={
    "clean_language": "language",
    "clean_original_length": "original_length",
    "clean_cleaned_length": "cleaned_length",
})[[
    "id", "cleaned_title", "cleaned_text", "authors", "category",
    "primary_category", "published", "language", "original_length", "cleaned_length",
]].rename(columns={"cleaned_title": "title", "cleaned_text": "text"})

OUT_PATH = Path("data") / "clean_arxiv_dataset.csv"
final_df.to_csv(OUT_PATH, index=False)

# Also save a parquet version (smaller, preserves dtypes -- convenient for Week 2)
try:
    final_df.to_parquet(Path("data") / "clean_arxiv_dataset.parquet", index=False)
    print("Parquet export saved (requires pyarrow).")
except Exception as e:
    print(f"Parquet export skipped ({e}) -- CSV is sufficient for Week 2.")

print(f"\nFinal clean dataset: {len(final_df):,} documents")
print(f"Saved to: {OUT_PATH}")
final_df.head(5)

In [ ]:
# Final sanity assertions before calling Week 1 complete
assert len(final_df) >= 5000, f"Only {len(final_df)} valid documents remain -- below the 5,000 minimum."
assert final_df["text"].isnull().sum() == 0
assert (final_df["text"].str.len() < 20).sum() == 0
assert final_df["id"].duplicated().sum() == 0

print("All Week 1 deliverable checks passed:")
print(f"  - {len(final_df):,} valid, cleaned, deduplicated documents")
print("  - Zero nulls in final text column")
print("  - Zero documents under the minimum length threshold")
print("  - Zero duplicate IDs")
print("\nWeek 1 complete. Dataset is ready for Week 2 (chunking & embedding).")

---
## Next Steps (Week 2 Preview)

- Chunk `data/clean_arxiv_dataset.csv` into overlapping passages (e.g. 300-500 tokens, ~15% overlap).
- Embed chunks with a sentence-transformer model and load them into **ChromaDB**.
- Build the retrieval + generation loop against **DeepSeek via OpenRouter**.

See `README.md` for full environment setup and run instructions.
